In [1]:
# Import PySpark tools
from pyspark.sql import SparkSession, functions as F

# Start Spark
spark = SparkSession.builder.appName("CS236_Data_Cleaning").getOrCreate()

# Reduce warning messages
spark.sparkContext.setLogLevel("ERROR")

# Load the NYC dataset
nyc_df = spark.read.csv("data/nyc_311_800k.csv", header=True, inferSchema=True)

# Load the LA dataset
la_df = spark.read.csv("data/la_311_800k.csv", header=True, inferSchema=True)

# Convert NYC dates to timestamps
nyc_clean = nyc_df.withColumn("created_ts", F.to_timestamp("created_date")).withColumn("closed_date", F.to_timestamp("closed_date"))

# Convert LA dates to timestamps
la_clean = la_df.withColumn("created_ts", F.to_timestamp("createddate")).withColumn("closeddate", F.to_timestamp("closeddate"))

# Keep source status unchanged and create the NYC cleaned status column
nyc_clean = nyc_clean.withColumn("cleaned_status", F.col("status"))

# Keep source status unchanged and create the LA cleaned status column
la_clean = la_clean.withColumn("cleaned_status", F.col("status"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 19:44:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. NYC and LA: closed dates, status, and resolution time

From EDA, we found out that there exists conflicts between the closed_date to start_date and status. Following are the checks of the count in each condition.

`status` preserves the source value; `cleaned_status` contains the final value after cleaning. All status changes and placeholder replacements apply only to `cleaned_status`.

In [2]:
# Flag NYC conditions and exclude cancellation, referral, and forwarding statuses
nyc_clean = (nyc_clean
    .withColumn("closed_without_date", F.coalesce((F.col("status") == "Closed") & F.col("closed_date").isNull(), F.lit(False)))
    .withColumn("closed_before_created", F.coalesce(F.col("closed_date") < F.col("created_ts"), F.lit(False)))
    .withColumn("status_changed_to_closed", F.col("closed_date").isNotNull() & ~F.coalesce(F.col("status").isin("Closed", "Cancelled", "Referred Out", "Forward"), F.lit(False))))

# Flag LA conditions and exclude cancellation, referral, and forwarding statuses
la_clean = (la_clean
    .withColumn("closed_without_date", F.coalesce((F.col("status") == "Closed") & F.col("closeddate").isNull(), F.lit(False)))
    .withColumn("closed_before_created", F.coalesce(F.col("closeddate") < F.col("created_ts"), F.lit(False)))
    .withColumn("status_changed_to_closed", F.col("closeddate").isNotNull() & ~F.coalesce(F.col("status").isin("Closed", "Cancelled", "Referred Out", "Forward"), F.lit(False))))

# Count each condition before cleaning
for city, df, closed_field in [("NYC", nyc_clean, "closed_date"), ("LA", la_clean, "closeddate")]:
    print(city)
    df.selectExpr(
        "sum(CAST(closed_without_date AS INT)) AS closed_without_date",
        "sum(CAST(closed_before_created AS INT)) AS reversed_dates",
        f"sum(CASE WHEN {closed_field} IS NOT NULL AND NOT COALESCE(status = 'Closed', FALSE) THEN 1 ELSE 0 END) AS date_without_closed_status",
        "sum(CAST(status_changed_to_closed AS INT)) AS eligible_status_changes",
        f"sum(CASE WHEN {closed_field} IS NOT NULL AND status IN ('Cancelled', 'Referred Out', 'Forward') THEN 1 ELSE 0 END) AS excluded_status_changes"
    ).show()

# Show which statuses have a closed date without Closed status
for city, df, closed_field in [("NYC", nyc_clean, "closed_date"), ("LA", la_clean, "closeddate")]:
    print(city)
    df.filter(F.col(closed_field).isNotNull() & ~F.coalesce(F.col("status") == "Closed", F.lit(False))).groupBy("status").count().orderBy(F.desc("count")).show()

NYC


+-------------------+--------------+--------------------------+-----------------------+-----------------------+
|closed_without_date|reversed_dates|date_without_closed_status|eligible_status_changes|excluded_status_changes|
+-------------------+--------------+--------------------------+-----------------------+-----------------------+
|                  4|           626|                     11109|                  11109|                      0|
+-------------------+--------------+--------------------------+-----------------------+-----------------------+

LA


+-------------------+--------------+--------------------------+-----------------------+-----------------------+
|closed_without_date|reversed_dates|date_without_closed_status|eligible_status_changes|excluded_status_changes|
+-------------------+--------------+--------------------------+-----------------------+-----------------------+
|               1991|             0|                     35643|                      0|                  35643|
+-------------------+--------------+--------------------------+-----------------------+-----------------------+

NYC


+-----------+-----+
|     status|count|
+-----------+-----+
|       Open| 7020|
|   Assigned| 2490|
|In Progress| 1535|
|Unspecified|   46|
|    Pending|   18|
+-----------+-----+

LA


+------------+-----+
|      status|count|
+------------+-----+
|   Cancelled|34711|
|Referred Out|  932|
+------------+-----+



### Checked conditions and cleaning decisions

| Condition in the original data | NYC | LA | Decision |
|---|---:|---:|---|
| Closed status, but no closed date | 4 | 1,991 | Keep the missing closed date as null; do not estimate it. |
| Closed date before creation, any status | 626 | 0 | Set closed date and duration to null. |
| Closed date exists, but status is not Closed | 11,109 | 35,643 | Exclude Cancelled, Referred Out, and Forward; change only remaining eligible statuses to Closed. |
| Eligible for status change after exclusions | 11,109 | 0 | Set status to Closed. |
| Excluded from status change despite a closed date | 0 | 35,643 | Keep the existing status. |


NYC's third condition includes Open (7,020), In Progress (1,535), Assigned (2,490), Pending (18), and Unspecified (46). LA's includes Cancelled (34,711) and Referred Out (932). NYC statuses listed above remain eligible for conversion to Closed. LA cancellations and referrals are excluded and keep their existing labels; all 164 Forward requests also keep their status and have no closed date. Status changes are written to cleaned_status, while status retains the source label; the source datasets remain unchanged.

### 1.1 Closed status, but no closed date

**Reason:** A Closed status does not tell us when closure happened, so we keep the missing date as null instead of inventing a closure time.

The 4 NYC and 1,991 LA requests remain Closed with null closed dates. No mean resolution time is calculated, and no closure dates are imputed.

In [3]:
# Leave missing closed dates as null
nyc_clean = nyc_clean.withColumn(
    "closed_date", F.when(F.col("closed_without_date"), F.lit(None).cast("timestamp")).otherwise(F.col("closed_date"))
)
la_clean = la_clean.withColumn(
    "closeddate", F.when(F.col("closed_without_date"), F.lit(None).cast("timestamp")).otherwise(F.col("closeddate"))
)

### 1.2 Closed date before creation, any status

**Reason:** A request cannot close before it is created, so we rewrite the invalid closure date to null instead of guessing a replacement.

In [4]:
# Set invalid closed timestamps to null
nyc_clean = nyc_clean.withColumn("closed_date", F.when(~F.col("closed_before_created"), F.col("closed_date")))
la_clean = la_clean.withColumn("closeddate", F.when(~F.col("closed_before_created"), F.col("closeddate")))

### 1.3 Closed date exists, but status is not Closed

**Reason:** A closure date can coexist with a cancellation, referral, or forwarding outcome, so we preserve those labels and apply the date-priority rule only to the remaining statuses.

Exclude `Cancelled`, `Referred Out`, and `Forward` before flagging requests for conversion to Closed. This leaves 11,109 NYC requests and 0 LA requests eligible; LA keeps all 34,725 Cancelled, 932 Referred Out, and 164 Forward requests. The rule for active or unspecified statuses remains a project assumption that a recorded closure date takes priority.

Write the result to `cleaned_status`; do not overwrite `status`.

In [5]:
# Set cleaned_status to Closed for the originally flagged requests
nyc_clean = nyc_clean.withColumn("cleaned_status", F.when(F.col("status_changed_to_closed"), "Closed").otherwise(F.col("cleaned_status")))
la_clean = la_clean.withColumn("cleaned_status", F.when(F.col("status_changed_to_closed"), "Closed").otherwise(F.col("cleaned_status")))

In [6]:
# Verify the three rules and row preservation in both cities
for city, df, closed_field, expected in [("NYC", nyc_clean, "closed_date", (4, 626, 11109)), ("LA", la_clean, "closeddate", (1991, 0, 0))]:
    counts = df.selectExpr(
        "count(*) AS rows",
        "sum(CAST(closed_without_date AS INT)) AS missing_closed",
        "sum(CAST(closed_before_created AS INT)) AS invalid",
        "sum(CAST(status_changed_to_closed AS INT)) AS changed",
        f"sum(CASE WHEN closed_without_date AND {closed_field} IS NOT NULL THEN 1 ELSE 0 END) AS failed_null_preservation",
        f"sum(CASE WHEN closed_before_created AND {closed_field} IS NOT NULL THEN 1 ELSE 0 END) AS failed_date_cleaning",
        "sum(CASE WHEN status_changed_to_closed AND cleaned_status != 'Closed' THEN 1 ELSE 0 END) AS failed_status_cleaning",
        f"sum(CASE WHEN {closed_field} IS NOT NULL AND NOT COALESCE(cleaned_status IN ('Closed', 'Cancelled', 'Referred Out', 'Forward'), FALSE) THEN 1 ELSE 0 END) AS remaining_status_conflicts",
    ).first()
    assert counts.rows == 800000
    assert (counts.missing_closed, counts.invalid, counts.changed) == expected
    # Check that missing dates stay null and invalid dates were cleared
    assert counts.failed_null_preservation == 0, f"{city}: missing closed dates were filled unexpectedly"
    assert counts.failed_date_cleaning == 0, f"{city}: invalid closed dates remain"

    # Check that only excluded statuses can retain a closed date without Closed cleaned_status
    assert counts.failed_status_cleaning == 0, f"{city}: flagged statuses were not updated"
    assert counts.remaining_status_conflicts == 0, f"{city}: eligible cleaned_status conflicts remain"
    print(city, "checks passed")
    

# Show the final NYC cleaned_status and date combinations
nyc_clean.groupBy("cleaned_status").agg(F.count("*").alias("requests"), F.count("closed_date").alias("with_closed_date"), F.sum(F.when(F.col("closed_date").isNull(), 1).otherwise(0)).alias("without_closed_date")).orderBy(F.desc("requests")).show()

# Show the final LA cleaned_status and date combinations
la_clean.groupBy("cleaned_status").agg(F.count("*").alias("requests"), F.count("closeddate").alias("with_closed_date"), F.sum(F.when(F.col("closeddate").isNull(), 1).otherwise(0)).alias("without_closed_date")).orderBy(F.desc("requests")).show()

# Check that excluded LA statuses were preserved
la_preserved_statuses = {row["cleaned_status"]: row["count"] for row in la_clean.filter(F.col("cleaned_status").isin("Cancelled", "Referred Out", "Forward")).groupBy("cleaned_status").count().collect()}
assert la_preserved_statuses == {"Cancelled": 34725, "Referred Out": 932, "Forward": 164}
print("LA cancellation, referral, and forwarding statuses preserved.")

# Check the final number of Closed requests without a closed date
assert nyc_clean.filter((F.col("cleaned_status") == "Closed") & F.col("closed_date").isNull()).count() == 630
assert la_clean.filter((F.col("cleaned_status") == "Closed") & F.col("closeddate").isNull()).count() == 1991
print("Closed requests with missing dates: NYC 630; LA 1,991.")

NYC checks passed


LA checks passed


+--------------+--------+----------------+-------------------+
|cleaned_status|requests|with_closed_date|without_closed_date|
+--------------+--------+----------------+-------------------+
|        Closed|  715898|          715268|                630|
|   In Progress|   53196|               0|              53196|
|          Open|   29741|               0|              29741|
|      Assigned|     832|               0|                832|
|       Pending|     312|               0|                312|
|       Started|      14|               0|                 14|
|   Unspecified|       7|               0|                  7|
+--------------+--------+----------------+-------------------+



+--------------+--------+----------------+-------------------+
|cleaned_status|requests|with_closed_date|without_closed_date|
+--------------+--------+----------------+-------------------+
|        Closed|  730126|          728135|               1991|
|     Cancelled|   34725|           34711|                 14|
|       Pending|   17972|               0|              17972|
|          Open|   16081|               0|              16081|
|  Referred Out|     932|             932|                  0|
|       Forward|     164|               0|                164|
+--------------+--------+----------------+-------------------+



LA cancellation, referral, and forwarding statuses preserved.


Closed requests with missing dates: NYC 630; LA 1,991.


### Final report justification

First, 4 NYC and 1,991 LA requests were marked Closed but had no closed date, so we left their dates null because the status does not tell us when closure happened. Second, 626 NYC requests had closed dates before creation, so we set those invalid dates to null rather than guess a correction. Third, 11,109 NYC and 35,643 LA requests had closed dates without Closed status, but we excluded Cancelled, Referred Out, and Forward because those labels describe distinct outcomes that can legitimately have a closure date. After this filter, we changed 11,109 NYC statuses and no LA statuses to Closed, preserving all cancellation, referral, and forwarding labels. All requests were retained; the final data contains 630 Closed NYC requests and 1,991 Closed LA requests with null closed dates, including the 17 NYC requests affected by both the invalid-date and status rules.

The source status is preserved in `status`, and the final status after these rules and placeholder cleaning is stored in `cleaned_status`.

## 2. NYC and LA: ZIP codes that are not five digits

In [7]:
# Show invalid NYC ZIP values and their counts
nyc_clean.filter(
    F.col("incident_zip").isNotNull() & ~F.col("incident_zip").rlike("^[0-9]{5}$")
).groupBy("incident_zip").count().show()

# Show invalid LA ZIP values and their counts
la_clean.filter(
    F.col("zipcode").isNotNull() & ~F.col("zipcode").rlike("^[0-9]{5}$")
).groupBy("zipcode").count().show()

+------------+-----+
|incident_zip|count|
+------------+-----+
|          No|    1|
+------------+-----+



+-------+-----+
|zipcode|count|
+-------+-----+
|      0|   38|
|  VE, 0|    1|
+-------+-----+



In [8]:
# Replace invalid values with null, keep valid ones as strings
nyc_clean = nyc_clean.withColumn(
    "incident_zip",
    F.when(F.col("incident_zip").cast("string").rlike("^[0-9]{5}$"),
           F.col("incident_zip").cast("string"))
)
la_clean = la_clean.withColumn(
    "zipcode",
    F.when(F.col("zipcode").cast("string").rlike("^[0-9]{5}$"),
           F.col("zipcode").cast("string"))
)

# Check that no invalid non-missing zip remained
assert nyc_clean.filter(F.col("incident_zip").isNotNull() & ~F.col("incident_zip").rlike("^[0-9]{5}$")).count() == 0
assert la_clean.filter(F.col("zipcode").isNotNull() & ~F.col("zipcode").rlike("^[0-9]{5}$")).count() == 0

### Final report explanation

We found one invalid NYC ZIP value (`No`) and 39 invalid LA ZIP values (38 of `0` and 1 of `VE, 0`). We replaced these values with null. All requests were retained, and existing missing ZIP values were left missing.

## 3. NYC: placeholder values in 'cleaned_status' and 'open_data_channel_type'

In [9]:
# Get unique cleaned_status values and their counts
nyc_clean.groupBy("cleaned_status").count().orderBy(F.desc("count")).show()

# Get unique submission channel values and their counts
nyc_clean.groupBy("open_data_channel_type").count().orderBy(F.desc("count")).show()

+--------------+------+
|cleaned_status| count|
+--------------+------+
|        Closed|715898|
|   In Progress| 53196|
|          Open| 29741|
|      Assigned|   832|
|       Pending|   312|
|       Started|    14|
|   Unspecified|     7|
+--------------+------+



+----------------------+------+
|open_data_channel_type| count|
+----------------------+------+
|                ONLINE|419261|
|                 PHONE|163519|
|                MOBILE|150744|
|               UNKNOWN| 65687|
|                 OTHER|   789|
+----------------------+------+



The source had 53 Unspecified statuses; section 1 changed 46 with recorded closure dates to Closed, leaving 7 Unspecified statuses. The submission channel still has 65,687 UNKNOWN values, with no existing nulls in either checked column. Replace these placeholders with null in cleaned_status and open_data_channel_type and keep all other values, including `OTHER`.

In [10]:
# Replace Unspecified cleaned_status with null
nyc_clean = nyc_clean.withColumn(
    "cleaned_status", F.when(F.col("cleaned_status") != "Unspecified", F.col("cleaned_status"))
)

# Replace UNKNOWN submission channel with null
nyc_clean = nyc_clean.withColumn(
    "open_data_channel_type",
    F.when(F.col("open_data_channel_type") != "UNKNOWN", F.col("open_data_channel_type"))
)

# Check that the placeholders are gone
assert nyc_clean.filter((F.col("cleaned_status") == "Unspecified") | (F.col("open_data_channel_type") == "UNKNOWN")).count() == 0

### Final report explanation

After section 1 reclassified 46 Unspecified statuses with closure dates, 7 Unspecified statuses remained, along with 65,687 UNKNOWN submission channel values. We replaced these placeholders with null in cleaned_status and open_data_channel_type because they do not identify a known status or channel. All requests and other category values, including `OTHER`, were retained.

## Final verification

Check the final row counts, request IDs, and missing-value totals. Status standardization is deferred until merging.

In [11]:
# Check final NYC row count, unique request IDs, and missing values
nyc_final = nyc_clean.selectExpr(
    "count(*) AS rows", "count(DISTINCT unique_key) AS unique_ids",
    "sum(CASE WHEN incident_zip IS NULL THEN 1 ELSE 0 END) AS missing_zip",
    "sum(CASE WHEN cleaned_status IS NULL THEN 1 ELSE 0 END) AS missing_status",
    "sum(CASE WHEN open_data_channel_type IS NULL THEN 1 ELSE 0 END) AS missing_channel"
).first()
assert nyc_final.rows == nyc_final.unique_ids == 800000
assert (nyc_final.missing_zip, nyc_final.missing_status, nyc_final.missing_channel) == (6882, 7, 65687)

# Check final LA row count, unique request IDs, and missing ZIPs
la_final = la_clean.selectExpr(
    "count(*) AS rows", "count(DISTINCT srnumber) AS unique_ids",
    "sum(CASE WHEN zipcode IS NULL THEN 1 ELSE 0 END) AS missing_zip"
).first()
assert la_final.rows == la_final.unique_ids == 800000
assert la_final.missing_zip == 926

# Show final verification results
print("NYC: 800,000 requests retained, unique IDs preserved, missing-value totals verified.")
print("LA: 800,000 requests retained, unique IDs preserved, missing ZIP total verified.")

# Check that the source status values are preserved in both cleaned datasets
for city, df, source, id_field in [
    ("NYC", nyc_clean, nyc_df, "unique_key"),
    ("LA", la_clean, la_df, "srnumber")
]:
    assert df.alias("clean").join(source.alias("source"), id_field).filter(
        ~F.col("clean.status").eqNullSafe(F.col("source.status"))
    ).count() == 0
    print(city, "source status unchanged; cleaned_status contains the cleaning results.")

NYC: 800,000 requests retained, unique IDs preserved, missing-value totals verified.
LA: 800,000 requests retained, unique IDs preserved, missing ZIP total verified.


NYC source status unchanged; cleaned_status contains the cleaning results.


LA source status unchanged; cleaned_status contains the cleaning results.
